In [ ]:
# Import Library

import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import random

In [23]:
# Load Data

bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

audio_file = 'sample_audio_speed.wav'
audio, sr = librosa.load(audio_file, sr=None) #librosa
clip = AudioSegment.from_wav(audio_file) #pydub

In [11]:
# Export Function

def export_function(audio):
    while True:
        file_name = input("Input Filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file name must ends with .wav")
    audio.export(file_name, format="wav")
    print("Audio saved as ", file_name)
    
def export_librosa(audio):
    while True:
        file_name = input("Input Filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("File name must ends with .wav")
    sf.write(file_name, np.clip(audio, -1,1), sr) # clip: noise/gain can exceed 1.0
    print("Audio saved as ", file_name)

In [9]:
# Predict Text Function
def predict_text(audio_file):
    waveform, sr = torchaudio.load(audio_file)
    waveform = waveform.mean(dim=0, keepdim=True) # stereo -> mono
    
    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)
        
    with torch.inference_mode():
        emission, _ = model(waveform)
        
    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0
            
        def forward(self, emission: torch.Tensor)->str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])
        
    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript

In [15]:
# Augmentation Function
# pydub (clip)      : volume +-, repeat, fade in, fade out, reverse
# librosa (audio)   : pitch, speed, white noise, random gain

def adjvolume(clip):
    while True:
        db = float(input("Input DB to adj volume (-30 to 30): "))
        if -30 <= db <=30:
            break
        else:
            print("Must be between -30 to 30")
    export_function(clip + db)
    
def repeat_clip(clip):
    while True:
        times = int(input("Input Repeat Audio Count (1 to 100): "))
        if 1 <= times <=100:
            break
        else:
            print("Must be between 1 to 100")
    export_function(clip * times)
    
def fade_in(clip):
    while True:
        duration = int(input("input fadein duration in ms (1s = 1000ms): "))
        if duration > 0:
            break
        else:
            print("must be more than 0")
    export_function(clip.fade_in(duration))
    
def fade_out(clip):
    while True:
        duration = int(input("input fadeout duration in ms (1s = 1000ms): "))
        if duration > 0:
            break
        else:
            print("must be more than 0")
    export_function(clip.fade_out(duration))
    
def reverse_audio(clip):
    export_function(clip.reverse())
    
def adjpitch():
    while True:
        pitch_rate = float(input("Input Pitch Rate (-5 to 5): "))
        if -5 <= pitch_rate <= 5:
            break
        else:
            print("must be between -5 to 5")
    shifted_audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_rate)
    export_librosa(shifted_audio)
    
def adjspeed():
    while True:
        rate = float(input("Input speed rate (Normal = 1)(0.5 to 2): "))
        if 0.5 <= rate <= 2:
            break
        else:
            print("must be between 0.5 to 2")
    speed_audio = librosa.effects.time_stretch(audio, rate=rate)
    export_librosa(speed_audio)
    
def white_noise():
    while True:
        level = float(input("Input noise level (0.1 to 0.5): "))
        if 0.1 <= level <= 0.5:
            break
        else:
            print("must be between 0.1 to 0.5")
    noise = np.random.normal(0, audio.std(), audio.size)
    export_librosa(audio + noise * level)
            
def random_gain():
    gain = np.random.uniform(0.5, 1.5)
    print(f"random gain x {gain:.2f}")
    export_librosa(audio * gain)


In [16]:
def aug_menu():
    while True:
        print("1. Volume")
        print("2. Repeat")
        print("3. Fade In")
        print("4. Fade Out")
        print("5. Reverse")
        print("6. Pitch")
        print("7. Speed")
        print("8. White Noise")
        print("9. Random Gain")
        print("10. Back to Main Menu")
        
        choice = input(">> ")
        if choice == "1":
            adjvolume(clip)
        elif choice == "2":
            repeat_clip(clip)
        elif choice == "3":
            fade_in(clip)
        elif choice == "4":
            fade_out(clip)
        elif choice == "5":
            reverse_audio(clip)
        elif choice == "6":
            adjpitch()
        elif choice == "7":
            adjspeed()
        elif choice == "8":
            white_noise()
        elif choice == "9":
            random_gain()
        elif choice == "10":
            break
        else:
            print("Invalid Input must be (1 to 10)")

In [24]:
def mainmenu():
    while True:
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")
        
        choice = input(">> ")
        if choice == "1":
            print("Audio Augmentation Menu")
            aug_menu()
        elif choice == "2":
            transcript = predict_text(audio_file)
            print("Hasil Transkrip: ")
            print(transcript)
        elif choice == "3":
            break
        else:
            print("Invalid Input")
    print("Main Menu Closed")

mainmenu()

1. Audio Augmentation
2. Predict Text
3. Exit
Hasil Transkrip: 
O ARE USE VERY NIGHTAR TES HEN IS ASKED CISSJISH ANY ORSES IS GA AN THAT ERY AGA PLACU SMARS BOOKS AND  IS ANR ON ZERIL SEES 
1. Audio Augmentation
2. Predict Text
3. Exit
Main Menu Closed
